In [1]:
from transformers import AutoTokenizer, AutoModel
from transformers import Wav2Vec2Processor, Wav2Vec2Model
from transformers import AutoImageProcessor
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_class_weight
from transformers import get_linear_schedule_with_warmup
import torch
import torch.nn as nn
import PIL.Image as Image
import numpy as np
import pandas as pd
import os
import librosa
import cv2

c:\Users\acer\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
text_tokenizer = AutoTokenizer.from_pretrained("C:/EmoBase/loaded resorce/emocontext_model2")
text_model = AutoModel.from_pretrained("C:/EmoBase/loaded resorce/emocontext_model2")

audio_processor = Wav2Vec2Processor.from_pretrained("C:/EmoBase/loaded resorce/wav2vec_model")
model_audio = Wav2Vec2Model.from_pretrained("C:/EmoBase/loaded resorce/wav2vec_model")

image_processor = AutoImageProcessor.from_pretrained("C:/EmoBase/loaded resorce/face_model")
model_image = AutoModel.from_pretrained("C:/EmoBase/loaded resorce/face_model")

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 3432.77it/s]
[transformers] RobertaModel LOAD REPORT from: C:/EmoBase/loaded resorce/emocontext_model2
Key                        | Status     | 
---------------------------+------------+-
classifier.dense.bias      | UNEXPECTED | 
classifier.dense.weight    | UNEXPECTED | 
classifier.out_proj.bias   | UNEXPECTED | 
classifier.out_proj.weight | UNEXPECTED | 
pooler.dense.weight        | MISSING    | 
pooler.dense.bias          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Loading weights: 100%|██████████| 211/211 [00:00<00:00, 3783.53it/s]
[transformers] Wav2Vec2Model LOAD REPORT from: C:/EmoBase/loaded resorce/wav2vec_model
Key               | Status     |  | 
------------------+------------+--+-
projector.bias    |

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using: {device}")

text_model = text_model.to(device).eval()
model_audio = model_audio.to(device).eval()
model_image = model_image.to(device).eval()

Using: cuda


In [4]:
df_train = pd.read_csv("C:/MELD.Raw/train/train_sent_emo.csv")
df_test = pd.read_csv("C:/MELD.Raw/test/test_sent_emo.csv")
df_dev = pd.read_csv("C:/MELD.Raw/dev/dev_sent_emo.csv")

label_map = {
    "neutral": 0, "joy": 1, "sadness": 2,
    "anger": 3, "fear": 4, "surprise": 5, "disgust": 6
}
df_train['label'] = df_train['Emotion'].map(label_map)
df_test['label'] = df_test['Emotion'].map(label_map)
df_dev['label'] = df_dev['Emotion'].map(label_map)

df_train = df_train.reset_index(drop=True)
df_dev = df_dev.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

In [5]:
def load_audio(file_path):
    try:
        y, sr = librosa.load(file_path, sr=16000)
        if len(y) == 0:
            return np.zeros(48000, dtype=np.float32)
        return y.astype(np.float32)
    except:
        return np.zeros(48000, dtype=np.float32)

def extract_frame(video_path):
    try:
        cap = cv2.VideoCapture(video_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        middle_frame = total_frames // 2
        cap.set(cv2.CAP_PROP_POS_FRAMES, middle_frame)
        ret, frame = cap.read()
        cap.release()
        if ret:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            return Image.fromarray(frame)
        else:
            return Image.new("RGB", (224, 224), color=(128, 128, 128))
    except:
        return Image.new("RGB", (224, 224), color=(128, 128, 128))

In [6]:
def extract_text_features(df, tokenizer, model, device, batch_size=32):
    model.eval()
    all_features = []
    with torch.no_grad():
        for i in range(0, len(df), batch_size):
            batch_df = df.iloc[i:i+batch_size]
            texts = batch_df['Utterance'].tolist()
            inputs = tokenizer(
                texts,
                max_length=128,
                padding="max_length",
                truncation=True,
                return_tensors="pt"
            )
            input_ids = inputs['input_ids'].to(device)
            attention_mask = inputs['attention_mask'].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            features = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            all_features.append(features)
            if i % 1000 == 0:
                print(f"Text: {i}/{len(df)}")
    return np.vstack(all_features)

In [7]:
def extract_audio_features(df, audio_folder, processor, model, device, batch_size=8):
    model.eval()
    all_features = []
    with torch.no_grad():
        for i in range(0, len(df), batch_size):
            batch_df = df.iloc[i:i+batch_size]
            batch_audio = []
            for _, row in batch_df.iterrows():
                path = os.path.join(audio_folder, f"dia{row['Dialogue_ID']}_utt{row['Utterance_ID']}.wav")
                audio = load_audio(path)
                batch_audio.append(audio)
            inputs = processor(
                batch_audio,
                sampling_rate=16000,
                return_tensors="pt",
                padding="max_length",
                max_length=48000,
                truncation=True
            )
            input_values = inputs['input_values'].to(device)
            outputs = model(input_values=input_values)
            features = outputs.last_hidden_state.mean(dim=1).cpu().numpy()
            all_features.append(features)
            if i % 500 == 0:
                print(f"Audio: {i}/{len(df)}")
    return np.vstack(all_features)

In [8]:
def extract_face_features(df, video_folder, processor, model, device, batch_size=32):
    model.eval()
    all_features = []
    with torch.no_grad():
        for i in range(0, len(df), batch_size):
            batch_df = df.iloc[i:i+batch_size]
            batch_frames = []
            for _, row in batch_df.iterrows():
                path = os.path.join(video_folder, f"dia{row['Dialogue_ID']}_utt{row['Utterance_ID']}.mp4")
                frame = extract_frame(path)
                batch_frames.append(frame)
            inputs = processor(images=batch_frames, return_tensors="pt")
            pixel_values = inputs['pixel_values'].to(device)
            outputs = model(pixel_values=pixel_values)
            features = outputs.pooler_output.squeeze(-1).squeeze(-1).cpu().numpy()
            all_features.append(features)
            if i % 500 == 0:
                print(f"Face: {i}/{len(df)}")
    return np.vstack(all_features)

In [9]:
os.makedirs("C:/EmoBase/features", exist_ok=True)

print("Extracting text features...")
train_text = extract_text_features(df_train, text_tokenizer, text_model, device)
dev_text = extract_text_features(df_dev, text_tokenizer, text_model, device)
test_text = extract_text_features(df_test, text_tokenizer, text_model, device)

print("Extracting audio features...")
train_audio = extract_audio_features(df_train, "C:/MELD.Raw/Audio/train", audio_processor, model_audio, device)
dev_audio = extract_audio_features(df_dev, "C:/MELD.Raw/Audio/dev", audio_processor, model_audio, device)
test_audio = extract_audio_features(df_test, "C:/MELD.Raw/Audio/test", audio_processor, model_audio, device)

print("Extracting face features...")
train_face = extract_face_features(df_train, "C:/MELD.Raw/train/train_splits", image_processor, model_image, device)
dev_face = extract_face_features(df_dev, "C:/MELD.Raw/dev/dev_splits_complete", image_processor, model_image, device)
test_face = extract_face_features(df_test, "C:/MELD.Raw/test/output_repeated_splits_test", image_processor, model_image, device)

print("Saving features...")
np.save("C:/EmoBase/features/train_text.npy", train_text)
np.save("C:/EmoBase/features/train_audio.npy", train_audio)
np.save("C:/EmoBase/features/train_face.npy", train_face)
np.save("C:/EmoBase/features/dev_text.npy", dev_text)
np.save("C:/EmoBase/features/dev_audio.npy", dev_audio)
np.save("C:/EmoBase/features/dev_face.npy", dev_face)
np.save("C:/EmoBase/features/test_text.npy", test_text)
np.save("C:/EmoBase/features/test_audio.npy", test_audio)
np.save("C:/EmoBase/features/test_face.npy", test_face)
print("All features saved!")

Extracting text features...
Text: 0/9989
Text: 4000/9989
Text: 8000/9989
Text: 0/1109
Text: 0/2610
Extracting audio features...
Audio: 0/9989
Audio: 1000/9989


C:\Users\acer\AppData\Local\Temp\ipykernel_21508\3086296077.py:3: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(file_path, sr=16000)
c:\Users\acer\.venv\lib\site-packages\librosa\core\audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


Audio: 2000/9989
Audio: 3000/9989
Audio: 4000/9989
Audio: 5000/9989
Audio: 6000/9989
Audio: 7000/9989
Audio: 8000/9989
Audio: 9000/9989
Audio: 0/1109
Audio: 1000/1109
Audio: 0/2610
Audio: 1000/2610
Audio: 2000/2610
Extracting face features...
Face: 0/9989
Face: 4000/9989
Face: 8000/9989
Face: 0/1109
Face: 0/2610
Saving features...
All features saved!


In [10]:
class FusionFeaturesDataset(Dataset):
    def __init__(self, text_features, audio_features, face_features, labels):
        self.text = torch.tensor(text_features, dtype=torch.float32)
        self.audio = torch.tensor(audio_features, dtype=torch.float32)
        self.face = torch.tensor(face_features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "text": self.text[idx],
            "audio": self.audio[idx],
            "face": self.face[idx],
            "labels": self.labels[idx]
        }


In [11]:
train_labels = df_train['label'].values
dev_labels = df_dev['label'].values
test_labels = df_test['label'].values

# ── Create datasets ──
train_dataset = FusionFeaturesDataset(train_text, train_audio, train_face, train_labels)
dev_dataset = FusionFeaturesDataset(dev_text, dev_audio, dev_face, dev_labels)
test_dataset = FusionFeaturesDataset(test_text, test_audio, test_face, test_labels)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
dev_loader = DataLoader(dev_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

print(len(train_dataset), len(dev_dataset), len(test_dataset))

9989 1109 2610


In [12]:
class SimpleFusionModel(nn.Module):
    def __init__(self, text_dim=768, audio_dim=768, face_dim=2048, num_classes=7):
        super(SimpleFusionModel, self).__init__()
        self.text_fc = nn.Linear(text_dim, 256)
        self.audio_fc = nn.Linear(audio_dim, 256)
        self.face_fc = nn.Linear(face_dim, 256)
        self.classifier = nn.Sequential(
            nn.Linear(256 * 3, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, text, audio, face):
        text_out = torch.relu(self.text_fc(text))
        audio_out = torch.relu(self.audio_fc(audio))
        face_out = torch.relu(self.face_fc(face))
        combined = torch.cat([text_out, audio_out, face_out], dim=1)
        return self.classifier(combined)

In [13]:
fusion_model = SimpleFusionModel().to(device)

y = train_labels
class_weight = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0,1,2,3,4,5,6]),
    y=y
)
class_weight_tensor = torch.tensor(class_weight, dtype=torch.float).to(device)
loss_fn = nn.CrossEntropyLoss(weight=class_weight_tensor)

In [38]:
total_steps = len(train_loader) * 60
warmup_steps = total_steps // 10
optimizer = torch.optim.AdamW(fusion_model.parameters(), lr=1e-5)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

# ── Training loop ──
for epoch in range(60):
    print(f"Epoch {epoch+1}/60")
    fusion_model.train()
    total_loss = 0

    for batch in train_loader:
        text = batch["text"].to(device)
        audio = batch["audio"].to(device)
        face = batch["face"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()
        outputs = fusion_model(text=text, audio=audio, face=face)
        loss = loss_fn(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(fusion_model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Average Loss: {avg_loss:.4f}")


Epoch 1/60
Average Loss: 0.0015
Epoch 2/60
Average Loss: 0.0013
Epoch 3/60
Average Loss: 0.0014
Epoch 4/60
Average Loss: 0.0013
Epoch 5/60
Average Loss: 0.0013
Epoch 6/60
Average Loss: 0.0011
Epoch 7/60
Average Loss: 0.0013
Epoch 8/60
Average Loss: 0.0012
Epoch 9/60
Average Loss: 0.0026
Epoch 10/60
Average Loss: 0.0011
Epoch 11/60
Average Loss: 0.0015
Epoch 12/60
Average Loss: 0.0015
Epoch 13/60
Average Loss: 0.0014
Epoch 14/60
Average Loss: 0.0014
Epoch 15/60
Average Loss: 0.0010
Epoch 16/60
Average Loss: 0.0010
Epoch 17/60
Average Loss: 0.0014
Epoch 18/60
Average Loss: 0.0016
Epoch 19/60
Average Loss: 0.0017
Epoch 20/60
Average Loss: 0.0013
Epoch 21/60
Average Loss: 0.0012
Epoch 22/60
Average Loss: 0.0018
Epoch 23/60
Average Loss: 0.0016
Epoch 24/60
Average Loss: 0.0014
Epoch 25/60
Average Loss: 0.0012
Epoch 26/60
Average Loss: 0.0011
Epoch 27/60
Average Loss: 0.0013
Epoch 28/60
Average Loss: 0.0014
Epoch 29/60
Average Loss: 0.0012
Epoch 30/60
Average Loss: 0.0008
Epoch 31/60
Average

In [20]:
torch.save(fusion_model.state_dict(), "C:/EmoBase/features/fusion_model.pth")
print("Fusion model saved!")

Fusion model saved!


In [ ]:
fusion_model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        text = batch["text"].to(device)
        audio = batch["audio"].to(device)
        face = batch["face"].to(device)
        labels = batch["labels"].to(device)
        outputs = fusion_model(text=text, audio=audio, face=face)
        preds = torch.argmax(outputs, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

target_names = ['neutral', 'joy', 'sadness', 'anger', 'fear', 'surprise', 'disgust']
print(classification_report(all_labels, all_preds, target_names=target_names))

              precision    recall  f1-score   support

     neutral       0.68      0.76      0.72      1256
         joy       0.45      0.51      0.48       402
     sadness       0.30      0.21      0.25       208
       anger       0.42      0.32      0.36       345
        fear       0.18      0.10      0.13        50
    surprise       0.44      0.41      0.42       281
     disgust       0.11      0.07      0.09        68

    accuracy                           0.55      2610
   macro avg       0.37      0.34      0.35      2610
weighted avg       0.53      0.55      0.54      2610



: 